# Data Setup

In [1]:
import pandas as pd

df=pd.read_csv(r'C:\Users\CSC\Documents\New Portfolio\AB testing\ab_testing_dataset.csv')
df.head()

,user_id,group,converted,timestamp
0,1,control,0,2024-01-01 00:00:00
1,2,treatment,0,2024-01-01 00:01:00
2,3,control,0,2024-01-01 00:02:00
3,4,control,1,2024-01-01 00:03:00
4,5,control,0,2024-01-01 00:04:00


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   user_id    50000 non-null  int64 
 1   group      50000 non-null  object
 2   converted  50000 non-null  int64 
 3   timestamp  50000 non-null  object
dtypes: int64(2), object(2)
memory usage: 1.5+ MB


In [8]:
df['user_id'].duplicated().sum()

0

In [10]:
df['group'].value_counts(normalize=True)

control      0.50092
treatment    0.49908
Name: group, dtype: float64

In [11]:
df['group'].unique()

array(['control', 'treatment'], dtype=object)

# Conversion Rate

In [12]:
conversion_rates=df.groupby('group')['converted'].mean()
conversion_rates

group
control      0.118183
treatment    0.130961
Name: converted, dtype: float64

In [16]:
control_rate=conversion_rates['control']
treatment_rate=conversion_rates['treatment']

In [17]:
print("Control:", control_rate)
print("Treatment:", treatment_rate)

Control: 0.11818254411882137
Treatment: 0.13096096818145386


# Lift

In [18]:
lift=treatment_rate-control_rate
print("Lift:", lift)

Lift: 0.012778424062632493


# Hypothesis testing (Z-test)

In [20]:
from statsmodels.stats.proportion import proportions_ztest

In [21]:
control=df[df['group']=='control']
treatment=df[df['group']=='treatment']

In [24]:
conversions=[control['converted'].sum(),treatment['converted'].sum()]
n=[len(control),len(treatment)]

z_stat,p_val=proportions_ztest(conversions,n,alternative='smaller')

print("Z-stat:", z_stat)
print("P-value:", p_val)

Z-stat: -4.326423958201133
P-value: 7.577480207909739e-06


# Decision

In [26]:
alpha = 0.05

if p_val < alpha:
    print("Reject null hypothesis: There is sufficient evidence that the treatment performs better than the control.")
else:
    print("Fail to reject null hypothesis: There is not enough evidence to conclude that the treatment performs better than the control.")

Reject null hypothesis: There is sufficient evidence that the treatment performs better than the control.


# Confidence Interval

In [29]:
import numpy as np

p1=control_rate
p2=treatment_rate

n1=len(control)
n2=len(treatment)

# standard error
se = np.sqrt((p1*(1-p1))/n1 + (p2*(1-p2))/n2)

z = 1.96

# margin of error
me = z * se

lower = (p2 - p1) - me
upper = (p2 - p1) + me

print("Confidence Interval:", (lower, upper))

Confidence Interval: (0.006990029149771999, 0.018566818975492987)
